# **KKBOX Churn Prediction And Retention Intelligence System - Data Preparation and Feature Ingineering**

## **1. Objectives**

## **2. Load Data & Basic Setup**

In [24]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


current_dir = os.getcwd()


while not os.path.exists(os.path.join(current_dir, "data", "raw")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/raw not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/raw exists:", os.path.exists("data/raw"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/raw exists: True


## **3. Light Cleaning**

In [25]:
# 3.1. Train dataset

train = pd.read_csv("data/raw/train_v2.csv")
train = train.drop_duplicates(subset="msno")

train["is_churn"] = train["is_churn"].astype(int)

In [26]:
# 3.2. Members dataset
members = pd.read_csv("data/raw/members_v3.csv")

members = members.drop_duplicates(subset="msno")

members.loc[(members["bd"] < 0) | (members["bd"] > 100), "bd"] = np.nan

members["gender"] = members["gender"].astype("category")

members["registration_init_time"] = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

members["city"] = pd.to_numeric(members["city"], errors="coerce")
members["registered_via"] = pd.to_numeric(members["registered_via"], errors="coerce")


In [27]:
# 3.3. Transactions dataset
transactions = pd.read_csv("data/raw/transactions_v2.csv")

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions = transactions[
    transactions["transaction_date"] <= transactions["membership_expire_date"]
]

num_cols = [
    "payment_method_id", "payment_plan_days", "plan_list_price",
    "actual_amount_paid", "is_auto_renew", "is_cancel"
]

transactions[num_cols] = transactions[num_cols].apply(pd.to_numeric, errors="coerce")

transactions.loc[transactions["actual_amount_paid"] < 0, "actual_amount_paid"] = np.nan
transactions.loc[transactions["plan_list_price"] < 0, "plan_list_price"] = np.nan
transactions.loc[transactions["payment_plan_days"] <= 0, "payment_plan_days"] = np.nan


In [28]:
# 3.4. User Logs dataset
user_logs = pd.read_csv("data/raw/user_logs_v2.csv")

log_cols = [
    "num_25", "num_50", "num_75", "num_985",
    "num_100", "num_unq", "total_secs"
]

user_logs[log_cols] = user_logs[log_cols].apply(pd.to_numeric, errors="coerce")

user_logs.loc[user_logs["total_secs"] < 0, "total_secs"] = np.nan
user_logs.loc[user_logs["num_unq"] < 0, "num_unq"] = np.nan

user_logs["date"] = pd.to_datetime(
    user_logs["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

user_logs = user_logs[
    (user_logs["date"] >= pd.Timestamp("2010-01-01")) &
    (user_logs["date"] <= pd.Timestamp("2020-01-01"))
]


In [ ]:
# 3.5. Quick Sanity Checks

print("Train:", train.shape)
print("Members:", members.shape)
print("Transactions:", transactions.shape)
print("User logs:", user_logs.shape)

print("Missing values in train:\n", train.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in members:\n", members.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in transactions:\n", transactions.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in user_logs:\n", user_logs.isnull().mean().sort_values(ascending=False).head(10))


print("Duplicate users in train:", train["msno"].duplicated().sum())
print("Duplicate users in members:", members["msno"].duplicated().sum())
print("Exact duplicate rows in transactions:", transactions.duplicated().sum())
print("Exact duplicate rows in user_logs:", user_logs.duplicated().sum())

train = train.reset_index(drop=True)
members = members.reset_index(drop=True)
transactions = transactions.reset_index(drop=True)
user_logs = user_logs.reset_index(drop=True)

Train: (970960, 2)
Members: (6769473, 6)
Transactions: (1425903, 9)
User logs: (18396362, 9)
Missing values in train:
 msno        0.0
is_churn    0.0
dtype: float64
Missing values in members:
 gender                    0.654335
bd                        0.000835
city                      0.000000
msno                      0.000000
registered_via            0.000000
registration_init_time    0.000000
dtype: float64
Missing values in transactions:
 payment_plan_days         0.001556
msno                      0.000000
payment_method_id         0.000000
plan_list_price           0.000000
actual_amount_paid        0.000000
is_auto_renew             0.000000
transaction_date          0.000000
membership_expire_date    0.000000
is_cancel                 0.000000
dtype: float64
Missing values in user_logs:
 msno          0.0
date          0.0
num_25        0.0
num_50        0.0
num_75        0.0
num_985       0.0
num_100       0.0
num_unq       0.0
total_secs    0.0
dtype: float64
Duplicate u